In [7]:
# import libraries
import time
import schedule, os
from datetime import datetime
# from time_diff_function import timediff
%run utilities.ipynb

Exception: File `'utilities.ipynb'` not found.

In [6]:
# an Eagle report lookup function, given six parameters
import datetime, time
%run utilities.ipynb

Exception: File `'utilities.ipynb'` not found.

In [1]:


#def osprey(rpt_type = 'r28i', funds = 'PABS', d_from = datetime.datetime.today(), d_to = datetime.datetime.today(), sfx = 'csv', al = 'qt', xe = 'dk'):
start_time          = time.time()

# (1) load libraries
#from datetime import datetime, timedelta
%run utilities.ipynb
import os
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
# https://stackoverflow.com/questions/38022658/selenium-python-handling-no-such-element-exception

# (2) set paths to the driver, urls, and to the report parameters
import os
os.environ["PATH"] = r'C:/SeleniumDrivers' # + os.pathsep + os.getenv("PATH")
# https://stackoverflow.com/questions/61213005/modify-beginning-of-path-variable-with-os-environ-in-python
pth                = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm' # user_defined variables stored here
url_default        = r'https://eagleportal.prescient.co.za/Default.aspx'
eagle_root         = r'https://eagleportal.prescient.co.za/Queries/Query.aspx?rpt='
# eagle report types, their short codes, and their URLs
report_types_dict = {'r28i': ['Reg 28 Report - Incl Effective Exposure', eagle_root + 'Reg28withExposure'],
                     'parn': ['Portfolio Analytics Report - New',        eagle_root + 'PortfolioAnalytics'],
                     'derv': ['Derivative Exposure',                     eagle_root + 'DerivativeExposure'],
                     'trad': ['Trades Report',                           eagle_root + 'TRANSACTION'],
                     'scty': ['Security Cross Reference',                eagle_root + 'SecurityCrossRef'],
                     'dflw': ['Daily Flows',                             eagle_root + 'FLOWS'],
                     'utps': ['Unit Trust Prices',                       eagle_root + 'UTPRICES'], 
                     'fnav': ['Fund Net Asset Value',                    eagle_root + 'NetAsset'],
                     'tcrf': ['Trades Cross Reference',                  eagle_root + 'TRADES%20REFERENCE']}  

Exception: File `'utilities.ipynb'` not found.

In [ ]:
# (3) prepare the report variables for fnds_, month_year_f, dayf, month_year_t, dayt,report_link
fnds_        = funds
month_year_f = f'{d_from:%B}, {d_from:%Y}' # e.g., 'January, 2023'
dayf         = f'{d_from:%#d}'             # e.g., '03', i.e., report day with a leading zero f'{d_from:%d}'
month_year_t = f'{d_to:%B}, {d_to:%Y}'     # e.g., 'January, 2023'
dayt         = f'{d_to:%#d}'               # e.g., '03', i.e., report day with a leading zero f'{d_from:%d}'
report_link  = report_types_dict[rpt_type][1]

# (4) assign the browser driver
from selenium import webdriver
driver = webdriver.Firefox()

# (5) open the browser on the Eagle web page
driver.get(url_default)          # default page
wait = WebDriverWait(driver, 10) # https://selenium-python.readthedocs.io/waits.html, max wait for elements to appear

# (6) login
driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_UserName'   ).send_keys(al)
driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_Password'   ).send_keys(xe)
driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_LoginButton').click()

# (7) having logged in, open the reporting page (NEEDS report type link)
driver.get(report_link)             # a hyperlink for the reporting page selected in the function osprey()

# (8) switch to the query page
driver.find_element(By.CSS_SELECTOR, '#ModifyLinkLabel').click()

# (8a) save the current window (NECESSARY?)
# edit_criteria_window = driver.window_handles[0] # save curent window handle
# https://stackoverflow.com/questions/10629815/how-to-switch-to-new-window-in-selenium-for-python

In [ ]:
# (9) update the FROM calendar with month_year_f and dayf
driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_B-1"]').click()
driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_NMC"]').click()
lmonth_selector = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 
                                                         'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_PMC"]')))
while driver.find_element(By.XPATH,'//td[@id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_TC"]').text != month_year_f:
    lmonth_selector.click()
day_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarDay"][text()={dayf}] | \
//td[@class="dxeCalendarDay dxeCalendarWeekend"][text()={dayf}]')
day_selector.click()

try:
    #https://stackoverflow.com/questions/38022658/selenium-python-handling-no-such-element-exception
    # (9A) update the TO calendar with month_year_t and dayt        
    driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_B-1"]').click() # date dropdown
    driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_NMC"]').click() # calendar
    rmonth_selector = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 
                                                             'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_PMC"]')))
    while driver.find_element(By.XPATH,'//td[@id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_To_DDD_C_TC"]').text != month_year_t:
        rmonth_selector.click()
    day_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarDay"][text()={dayt}] | \
    //td[@class="dxeCalendarDay dxeCalendarWeekend"][text()={dayt}]')
    day_selector.click()
except NoSuchElementException:
    pass
        
# (10) get the web element for the FUND LIST and assign values to it
fund_selector  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedIds"]')
driver.execute_script(f'arguments[0].value = "{fnds_}";', fund_selector)

# TEST ---------------------
fund_selector2  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_BtnAdvancedSearch"]')
driver.execute_script(f'arguments[0].value = "{fnds_}";', fund_selector2)

# TEST ---------------------

# (11) get the web element of the 'Submit' button and then click it
submit_button  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_RunBtn"]')
submit_button.click()

# (ALTERNATIVE to 11)
# wait for the 'Submit' button to be clickable and then click it
# WebDriverWait(driver, 100).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_RunBtn"]'))).click()

# (12) Wait for and then click the export button and then the xls download button
#https://stackoverflow.com/questions/56085152/selenium-python-error-element-could-not-be-scrolled-into-view
WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR,   'a[id="DistrBtn"]'        ))).click()
t = "0" if sfx == 'csv' else "4" # DXI4(0) for .xls(.csv)
WebDriverWait(driver, 1000).until(EC.element_to_be_clickable((By.CSS_SELECTOR, f'td[id="ExportMnu_DXI{t}_T"]'))).click()

time.sleep(5) # wait for 5 seconds after the data downloads

print(f'Downloading and then saving the {rpt_type} report in {sfx} format for {len(fnds_.split(","))} funds: {timediff(start_time, time.time())}', '\n')

# (13) having downloaded the requested report, close the web driver
driver.quit()
#print(f'Roundtrip time for getting holdings and derivative data: {timediff(start_time_overlord, time.time())}', '\n')

print(f'Roundtrip time to run the report lookup function: {timediff(start_time, time.time())}', '\n')